## Gemini API를 활용한 제품 리뷰 비교 분석



**목표:**
1. 동일 품목(선크림)의 다양한 제품 리뷰 수집 및 분석
2. 제품별 장단점 자동 추출
3. 경쟁사 제품 비교 분석
4. 시각화를 통한 인사이트 도출
5. Gemini를 활용한 종합 인사이트 도출

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly scikit-learn google-genai pydantic python-dotenv wordcloud nbformat kaleido

In [ ]:
# 필요한 라이브러리

import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any
import enum
from tqdm import tqdm
from datetime import datetime
from pprint import pprint
import time

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash-lite')

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {gemini_model}")

# 클라이언트 초기화
client = genai.Client(api_key=api_key)

 ### 1. 제품 리뷰 데이터 로드 및 전처리

In [ ]:
import pandas as pd
import json
import glob

# 제품 리뷰 데이터 로드
print("제품 리뷰 데이터 로드 중...")

json_files = glob.glob("./data/product_*_reviews.json")

if not json_files:
    print("product_*_reviews.json 파일을 찾을 수 없습니다.")
else:
    all_reviews = []
    product_info = {}
    
    for file_path in json_files:
        print(f"  {file_path} 로드 중...")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            product_id = data['product_id']
            total_reviews = data['total_reviews']
            reviews = data['reviews']
            
            # 제품 정보 저장
            if reviews:
                product_name = reviews[0]['product_name']
                vendor_name = reviews[0]['vendor_name']
                product_info[product_id] = {
                    'name': product_name,
                    'vendor': vendor_name,
                    'total_reviews': total_reviews,
                    'loaded_reviews': len(reviews)
                }
                
                print(f"    제품명: {product_name[:50]}...")
                print(f"    리뷰 수: {len(reviews)}개")
            
            # 리뷰 데이터 추가
            for review in reviews:
                review['source_file'] = file_path
                all_reviews.append(review)
                
        except Exception as e:
            print(f"    오류: {file_path} 로드 실패 - {e}")
    
    # DataFrame 생성
    reviews_df = pd.DataFrame(all_reviews)
    
    print(f"\n데이터 로드 완료:")
    print(f"  - 제품 수: {len(product_info)}개")
    print(f"  - 총 리뷰 수: {len(reviews_df)}개")
    
    # 제품별 통계
    print(f"\n제품별 리뷰 통계:")
    for product_id, info in product_info.items():
        avg_rating = reviews_df[reviews_df['product_id'] == product_id]['rating'].mean()
        print(f"  - {info['name'][:30]}...: {info['loaded_reviews']}개 리뷰, 평균 {avg_rating:.2f}점")
    
    # 데이터 전처리
    print(f"\n데이터 전처리 중...")
    
    initial_count = len(reviews_df)
    reviews_df = reviews_df.dropna(subset=['review_content'])
    reviews_df = reviews_df[reviews_df['review_content'].str.strip() != '']
    reviews_df = reviews_df.reset_index(drop=True)
    
    print(f"  - 전처리 후: {len(reviews_df)}개 ({initial_count - len(reviews_df)}개 제거)")
    
    # 리뷰 길이 분석
    reviews_df['review_length'] = reviews_df['review_content'].str.len()
    print(f"  - 평균 리뷰 길이: {reviews_df['review_length'].mean():.1f}자")
    
    # 평점 분포
    print(f"\n전체 평점 분포:")
    rating_dist = reviews_df['rating'].value_counts().sort_index()
    for score, count in rating_dist.items():
        percentage = (count / len(reviews_df)) * 100
        print(f"  {score}점: {count}개 ({percentage:.1f}%)")
    
    display(reviews_df.head())

 ### 2. 제품 리뷰 분석 응답 스키마 정의

In [ ]:
class ProductAspect(str, enum.Enum):
    QUALITY = "quality"                    # 품질
    USABILITY = "usability"                # 사용성/편의성
    FUNCTIONALITY = "functionality"        # 기능성/성능
    DESIGN = "design"                      # 디자인/외관
    PRICE_VALUE = "price_value"            # 가성비
    DURABILITY = "durability"              # 내구성/지속성
    CUSTOMER_SERVICE = "customer_service"  # 고객서비스
    SHIPPING = "shipping"                  # 배송/주문
    PACKAGING = "packaging"                # 포장/용기
    OVERALL = "overall"                    # 전반적 만족도

class SentimentType(str, enum.Enum):
    VERY_POSITIVE = "very_positive"        # 매우 긍정
    POSITIVE = "positive"                  # 긍정
    NEUTRAL = "neutral"                    # 중립
    NEGATIVE = "negative"                  # 부정
    VERY_NEGATIVE = "very_negative"        # 매우 부정

class AspectAnalysis(BaseModel):
    aspect: ProductAspect = Field(description="분석 측면")
    sentiment: SentimentType = Field(description="해당 측면에 대한 감정")
    keywords: List[str] = Field(description="관련 키워드", max_length=3)
    comment: str = Field(description="구체적 의견", max_length=200)
    confidence: float = Field(ge=0.0, le=1.0, description="분석 신뢰도")

class ProductReviewAnalysis(BaseModel):
    # 원본 정보
    review_id: str = Field(description="리뷰 ID")
    product_name: str = Field(description="제품명")
    rating: int = Field(ge=1, le=5, description="평점")
    review_text: str = Field(description="원본 리뷰 텍스트")
    
    # 측면별 분석
    aspect_analyses: List[AspectAnalysis] = Field(description="측면별 분석", max_length=8)
    
    # 종합 분석
    overall_sentiment: SentimentType = Field(description="전체 감정")
    pros: List[str] = Field(description="장점", max_length=5)
    cons: List[str] = Field(description="단점", max_length=5)
    summary: str = Field(description="리뷰 요약", max_length=300)
    recommendation_score: float = Field(ge=0.0, le=1.0, description="추천도")
    
class BatchProductReviewAnalysis(BaseModel):
    """여러 제품 리뷰를 배치로 분석한 결과"""
    reviews: List[ProductReviewAnalysis] = Field(description="분석된 리뷰 목록")
    total_count: int = Field(description="전체 리뷰 수")
    
    
print("✅ 제품 리뷰 분석 스키마 정의 완료")

 ### 3. AI 리뷰 분석 함수

In [ ]:
system_instruction = """
# 당신은 제품 리뷰 전문 분석가입니다.

## 분석해야 할 10가지 범용 제품 측면:
1. QUALITY: 제품의 품질, 만듦새, 재질, 완성도
2. USABILITY: 사용 편의성, 사용법의 간편함, 직관성, UX
3. FUNCTIONALITY: 기능/성능, 목적 달성도, 효과성, 작동
4. DESIGN: 디자인, 외관, 미적 만족도, 스타일, 색상
5. PRICE_VALUE: 가성비, 가격 대비 만족도, 경제성
6. DURABILITY: 내구성, 지속성, 장기 사용 가능성
7. CUSTOMER_SERVICE: 고객 서비스, A/S, 문의 응답
8. SHIPPING: 배송, 주문 과정, 배송 속도/상태
9. PACKAGING: 제품 포장, 용기, 개봉 경험
10. OVERALL: 전반적 만족도, 재구매 의향

## 감정 분류 기준 (SentimentType):
- VERY_POSITIVE (0.8~1.0): "최고", "완벽", "강력추천", "대만족"
- POSITIVE (0.6~0.8): "좋다", "만족", "괜찮다", "추천"
- NEUTRAL (0.4~0.6): "보통", "그저그렇다", "나쁘지않다"
- NEGATIVE (0.2~0.4): "아쉽다", "별로", "실망"
- VERY_NEGATIVE (0.0~0.2): "최악", "끔찍", "비추천", "환불"

## 신뢰도 (Confidence) 기준:
- 0.9~1.0: 구체적이고 명확한 언급 (예: "배송이 하루만에 도착")
- 0.7~0.9: 일반적 언급 (예: "배송 빨랐음")
- 0.5~0.7: 간접적 언급 (예: "빨리 받음")
- 0.3~0.5: 유추 가능한 언급 (예: "금방 옴")
- 0.1~0.3: 매우 모호한 언급

## 키워드 추출 원칙:
- 리뷰 **원문에서 사용된 단어** 우선
- **명사/동사** 중심 (예: "빠른 배송" → "배송속도", "잘 작동" → "작동")
- 구체적 표현 우선 (예: "좋음" → "견고함")
- 중요도 순으로 최대 3개

## 추천도 (Recommendation Score):
**아래 요소를 종합 판단하여 점수 산출:**

1. **평점 기반 (기본값):**
   - 5점 → 0.9~1.0 | 4점 → 0.7~0.9 | 3점 → 0.5~0.7 | 2점 → 0.3~0.5 | 1점 → 0.1~0.3

2. **조정 요소:**
   구체적이고 신뢰할 수 있는 리뷰에 대해서 아래의 점수 부여
   - 재구매/추천 명시: ±0.1~0.2
   - 매우 긍정/부정 표현 다수: ±0.1~0.15
   - **평점-내용 불일치 시: 내용 우선**

**최종 점수:**
- 0.8~1.0: 강력 추천 | 0.6~0.8: 추천 | 0.4~0.6: 보통 | 0.2~0.4: 비추천 | 0.0~0.2: 강력 비추천

## Summary 작성:
- 1~2문장, 300자 이내
- "평점-핵심 측면-종합 의견" 형식
- 예: "5점. 배송 빠르고 품질 우수. 가성비 좋아 재구매 의향 있음."

## 분석 원칙:
1. **명시적 언급 우선**: 직접 언급된 내용만 분석
2. **추측 금지**: 추론 내용은 신뢰도 낮게 처리
3. **객관성**: 실제 사용된 단어/표현 추출
4. **일관성**: 평점-텍스트 불일치 시 텍스트 우선
5. **범용성**: 모든 제품 카테고리 적용 가능

**언급 안 된 측면은 제외하고, 최대 8개 측면까지만 분석하세요.**
"""

In [ ]:
def analyze_product_review(review_data: dict, max_retries=2) -> Optional[ProductReviewAnalysis]:
    """
    개별 제품 리뷰를 AI로 분석 (오류 메시지 피드백 기반 재시도)
    """
    
    # 기본 프롬프트
    base_prompt = f"""
제품 리뷰를 종합적으로 분석해주세요:

리뷰 ID: {review_data['review_id']}
제품명: {review_data['product_name']}
평점: {review_data['rating']}점
리뷰 내용: "{review_data['review_content']}"
"""
    
    # API 호출 설정
    generation_config = types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type='application/json',
        response_schema=ProductReviewAnalysis,
        system_instruction=system_instruction
    )
    
    # 오류 메시지 저장
    last_error_message = None
    
    # 재시도 루프
    for attempt in range(max_retries + 1):
        try:
            # 프롬프트 구성 (오류 메시지 포함)
            if last_error_message:
                prompt = base_prompt + f"""

**재시도 {attempt}회 - 이전 응답에서 오류 발생**:
{last_error_message}

위 오류를 수정하여 다시 응답해주세요!
"""
            else:
                prompt = base_prompt
            
            # API 호출
            response = client.models.generate_content(
                model=gemini_model,
                contents=prompt,
                config=generation_config
            )
            
            # 파싱 시도
            if response.parsed:
                return response.parsed
            else:
                # 텍스트에서 JSON 추출
                response_text = response.text
                
                # JSON 파싱
                review_analysis_data = json.loads(response_text)
                
                # Pydantic 모델로 변환
                return ProductReviewAnalysis(**review_analysis_data)
        
        except Exception as e:
            # 오류 메시지 저장
            last_error_message = str(e)
            
            if attempt < max_retries:
                print(f"  ⚠️  오류 발생 (시도 {attempt + 1}/{max_retries + 1}), 재시도 중...")
                print(f"     리뷰 ID: {review_data.get('review_id', 'Unknown')}")
                print(f"     오류: {str(e)[:150]}...")
                time.sleep(1)
                continue
            else:
                print(f"  ❌ 리뷰 분석 실패")
                print(f"     리뷰 ID: {review_data.get('review_id', 'Unknown')}")
                print(f"     오류: {str(e)[:150]}...")
                return None
    
    return None

print("✅ AI 리뷰 분석 함수 준비 완료")

In [ ]:
def analyze_product_reviews_batch(reviews_data: List[dict], max_retries=2) -> Optional[BatchProductReviewAnalysis]:
    """
    여러 제품 리뷰를 배치로 AI 분석 (오류 메시지 피드백 기반 재시도)
    """
    
    # 기본 프롬프트 구성
    reviews_text = ""
    for idx, review in enumerate(reviews_data, 1):
        reviews_text += f"""
[리뷰 {idx}]
- 리뷰 ID: {review['review_id']}
- 제품명: {review['product_name']}
- 평점: {review['rating']}점
- 리뷰 내용: "{review['review_content']}"
"""
    
    base_prompt = f"""
다음 {len(reviews_data)}개의 제품 리뷰를 개별적으로 분석하세요.

{reviews_text}
"""
    
    # API 호출 설정
    generation_config = types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type='application/json',
        response_schema=BatchProductReviewAnalysis,
        system_instruction=system_instruction
    )
    
    # 오류 메시지 저장
    last_error_message = None
    
    # 재시도 루프
    for attempt in range(max_retries + 1):
        try:
            # 프롬프트 구성 (오류 메시지 포함)
            if last_error_message:
                prompt = base_prompt + f"""

**재시도 {attempt}회 - 이전 응답에서 오류 발생**:
{last_error_message}

위 오류를 수정하여 다시 응답해주세요!
특히 다음 사항을 확인해주세요:
- 모든 리뷰가 포함되었는지 (총 {len(reviews_data)}개)
- 각 필드의 데이터 타입과 제약조건이 올바른지
- review_id, product_name, rating, review_text가 원본 데이터와 일치하는지
"""
            else:
                prompt = base_prompt
            
            # API 호출
            response = client.models.generate_content(
                model=gemini_model,
                contents=prompt,
                config=generation_config
            )
            
            # 파싱 시도
            if response.parsed:
                result = response.parsed
                
                # 검증: 리뷰 개수 확인
                if len(result.reviews) != len(reviews_data):
                    raise ValueError(
                        f"리뷰 개수 불일치: 예상 {len(reviews_data)}개, 실제 {len(result.reviews)}개"
                    )
                
                return result
            else:
                # 텍스트에서 JSON 추출
                response_text = response.text
                
                # JSON 파싱
                batch_data = json.loads(response_text)
                
                # Pydantic 모델로 변환
                result = BatchProductReviewAnalysis(**batch_data)
                
                # 검증: 리뷰 개수 확인
                if len(result.reviews) != len(reviews_data):
                    raise ValueError(
                        f"리뷰 개수 불일치: 예상 {len(reviews_data)}개, 실제 {len(result.reviews)}개"
                    )
                
                return result
        
        except Exception as e:
            # 오류 메시지 저장
            last_error_message = str(e)
            
            if attempt < max_retries:
                print(f"  ⚠️  배치 분석 오류 발생 (시도 {attempt + 1}/{max_retries + 1}), 재시도 중...")
                print(f"     리뷰 개수: {len(reviews_data)}개")
                print(f"     오류: {str(e)[:200]}...")
                time.sleep(2)  # 배치는 조금 더 긴 대기 시간
                continue
            else:
                print(f"  ❌ 배치 분석 실패")
                print(f"     리뷰 개수: {len(reviews_data)}개")
                print(f"     오류: {str(e)[:200]}...")
                return None
    
    return None

print("✅ AI 리뷰 배치 분석 함수 준비 완료")

 ### 4. 제품별 리뷰 분석 실행

In [ ]:
# =============================================================================
# 제품 리뷰 배치 분석 실행
# =============================================================================

print(" 제품 리뷰 AI 배치 분석 시작...")
print(f"📝 전체 리뷰 수: {len(reviews_df)}개")

# 배치 설정
batch_size = 3
total_batches = (len(reviews_df) + batch_size - 1) // batch_size

analysis_results = []

# 배치별 처리
for batch_num in range(total_batches):
    start_idx = batch_num * batch_size
    end_idx = min(start_idx + batch_size, len(reviews_df))
    batch_df = reviews_df.iloc[start_idx:end_idx]
    
    print(f"\n 배치 {batch_num + 1}/{total_batches} ({len(batch_df)}개)")
    
    # 배치 데이터 준비
    batch_reviews_data = []
    for idx, review in batch_df.iterrows():
        batch_reviews_data.append({
            'review_id': str(review['review_id']),
            'product_name': review['product_name'],
            'rating': review['rating'],
            'review_content': review['review_content']
        })
    
    # 배치 분석 시도
    batch_result = analyze_product_reviews_batch(batch_reviews_data, max_retries=2)
    
    if batch_result:
        # 배치 성공
        print(f"  ✅ 배치 분석 완료: {len(batch_result.reviews)}개")
        for review_analysis in batch_result.reviews:
            result_dict = review_analysis.model_dump()
            # review_id를 문자열로 변환하여 비교
            original_review = batch_df[batch_df['review_id'].astype(str) == str(review_analysis.review_id)].iloc[0]
            result_dict['product_id'] = original_review['product_id']
            analysis_results.append(result_dict)
    else:
        # 배치 실패 → 개별 처리
        print(f"  ⚠️  배치 실패 → 개별 처리")
        for review_data in batch_reviews_data:
            individual_result = analyze_product_review(review_data, max_retries=2)
            if individual_result:
                result_dict = individual_result.model_dump()
                # review_id를 문자열로 변환하여 비교
                original_review = batch_df[batch_df['review_id'].astype(str) == str(review_data['review_id'])].iloc[0]
                result_dict['product_id'] = original_review['product_id']
                analysis_results.append(result_dict)
                print(f"    ✅ {review_data['review_id']}")
            else:
                print(f"    ❌ {review_data['review_id']}")
            time.sleep(2)
    
    time.sleep(3)

print(f"\n분석 완료: {len(analysis_results)}/{len(reviews_df)}개")

# DataFrame 변환
analysis_df = pd.DataFrame(analysis_results)

In [ ]:
# 데이터 샘플 확인
print("\n📊 분석 결과 샘플 확인")
print("=" * 60)
print(f"총 {len(analysis_df)}개 리뷰 분석 완료\n")

# 상위 5개 샘플 출력
for idx, row in analysis_df.head().iterrows():
    print(f"\n[샘플 {idx + 1}]")
    print(f"제품명: {row['product_name']}")
    print(f"평점: {row['rating']}점")
    print(f"전체 감정: {row['overall_sentiment']}")
    print(f"추천도: {row['recommendation_score']:.2f}")
    print(f"장점: {row['pros']}")
    print(f"단점: {row['cons']}")
    print("-" * 60)

#### 결과 저장하기

In [ ]:
# =============================================================================
# 분석 결과 JSON 저장
# =============================================================================

from datetime import datetime

# 타임스탬프 생성
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"product_review_analysis_{timestamp}.json"

# DataFrame을 JSON으로 저장
analysis_df.to_json(filename, orient='records', force_ascii=False, indent=2)

print(f"\n💾 분석 결과 저장 완료")
print(f"   파일명: {filename}")
print(f"   저장된 리뷰 수: {len(analysis_df)}개")